<a href="https://colab.research.google.com/github/lokeshwaran-rp/-MENUMANAGER-Menu-Q-A-Assistant/blob/main/MENUMANAGER_Menu_Q%26A_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y transformers sentence-transformers peft accelerate
!pip install -q transformers==4.39.3
!pip install -q sentence-transformers==2.7.0
!pip install -q accelerate==0.29.3

Found existing installation: transformers 5.12.0
Uninstalling transformers-5.12.0:
  Successfully uninstalled transformers-5.12.0
Found existing installation: sentence-transformers 5.5.1
Uninstalling sentence-transformers-5.5.1:
  Successfully uninstalled sentence-transformers-5.5.1
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 12.6 MB/s eta 0:00:00


In [2]:
import transformers
import sentence_transformers
import accelerate

print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 4.39.3
Sentence Transformers: 2.7.0
Accelerate: 0.29.3


In [3]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("Imports Successful")

Imports Successful


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/menu.csv')

print(df.shape)
df.head()

(117, 5)


,dish_name,category,price,description,dietary_tags
0,Paneer Tikka,Starter,260,Char-grilled cubes of fresh paneer marinated o...,"Veg, Spicy, Contains Dairy"
1,Paneer 65,Starter,240,Crispy fried paneer cubes tossed in a fiery So...,"Veg, Spicy, Contains Dairy"
2,Paneer Chilli Dry,Starter,250,Golden fried paneer cubes wok-tossed with caps...,"Veg, Spicy, Contains Dairy"
3,Veg Manchurian Dry,Starter,220,Crispy vegetable and cabbage dumplings tossed ...,"Veg, Spicy"
4,Gobi Manchurian,Starter,220,Crispy battered cauliflower florets tossed in ...,"Veg, Spicy"


In [6]:
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

Index(['dish_name', 'category', 'price', 'description', 'dietary_tags'], dtype='object')

Missing Values:
dish_name       0
category        0
price           0
description     0
dietary_tags    0
dtype: int64


In [7]:
df["combined_text"] = (
    df["dish_name"].astype(str)
    + " | "
    + df["category"].astype(str)
    + " | "
    + df["dietary_tags"].astype(str)
    + " | ₹"
    + df["price"].astype(str)
    + " | "
    + df["description"].astype(str)
)

df[["dish_name", "combined_text"]].head()

,dish_name,combined_text
0,Paneer Tikka,"Paneer Tikka | Starter | Veg, Spicy, Contains ..."
1,Paneer 65,"Paneer 65 | Starter | Veg, Spicy, Contains Dai..."
2,Paneer Chilli Dry,"Paneer Chilli Dry | Starter | Veg, Spicy, Cont..."
3,Veg Manchurian Dry,"Veg Manchurian Dry | Starter | Veg, Spicy | ₹2..."
4,Gobi Manchurian,"Gobi Manchurian | Starter | Veg, Spicy | ₹220 ..."


In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


In [9]:
dish_embeddings = embedding_model.encode(
    df["combined_text"].tolist(),
    convert_to_numpy=True
)

print("Embedding Shape:", dish_embeddings.shape)

Embedding Shape: (117, 384)


In [10]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def search(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    similarities = cosine_similarity(
        query_embedding,
        dish_embeddings
    )[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = df.iloc[top_indices].copy()

    results["similarity_score"] = similarities[top_indices]

    return results[
        [
            "dish_name",
            "category",
            "price",
            "dietary_tags",
            "similarity_score"
        ]
    ]

In [11]:
search("healthy food")

,dish_name,category,price,dietary_tags,similarity_score
89,Green Salad,Salad,120,"Veg, Vegan",0.433668
94,Sprouts Salad,Salad,140,"Veg, Vegan",0.416423
75,Veg Fried Rice,Rice,220,"Veg, Vegan",0.363188
83,Hot and Sour Soup (Veg),Soup,160,"Veg, Vegan, Spicy",0.362771
91,Boondi Raita,Salad,110,"Veg, Contains Dairy",0.359583


In [12]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)

print("QA Model Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

QA Model Loaded


In [13]:
def ask(question):

    matches = search(question, top_k=3)

    best_match = matches.iloc[0]["dish_name"]

    context = df[
        df["dish_name"] == best_match
    ]["combined_text"].values[0]

    answer = qa_pipeline(
        question=question,
        context=context
    )

    return {
        "Top Matches": matches,
        "Answer": answer["answer"],
        "Confidence": answer["score"]
    }

In [14]:
result = ask("Is butter chicken spicy?")

print("Answer:", result["Answer"])
print("Confidence:", result["Confidence"])

display(result["Top Matches"])

Answer: tomato-butter gravy with cashew paste and fresh cream
Confidence: 0.03573613986372948


,dish_name,category,price,dietary_tags,similarity_score
41,Butter Chicken,Main Course,360,"Non-Veg, Contains Dairy",0.632224
24,Chilli Chicken Dry,Starter,320,"Non-Veg, Spicy",0.619532
13,Chicken 65,Starter,300,"Non-Veg, Spicy",0.565488


In [15]:
def menu_qa(question):

    result = ask(question)

    print("\nQUESTION:")
    print(question)

    print("\nDIRECT ANSWER:")
    print(result["Answer"])

    print("\nCONFIDENCE:")
    print(round(result["Confidence"], 4))

    print("\nTOP MATCHES:")

    display(result["Top Matches"])

In [16]:
menu_qa("Suggest a light and nutritious meal")


QUESTION:
Suggest a light and nutritious meal

DIRECT ANSWER:
kebabs and curries

CONFIDENCE:
0.0733

TOP MATCHES:


,dish_name,category,price,dietary_tags,similarity_score
93,Onion Salad,Salad,90,"Veg, Vegan, Spicy",0.440189
89,Green Salad,Salad,120,"Veg, Vegan",0.433110
90,Kachumber Salad,Salad,130,"Veg, Vegan",0.398723


In [17]:
while True:

    question = input("\nAsk Menu Question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    menu_qa(question)


Ask Menu Question: exit
Goodbye!


In [18]:
import re

def filtered_search(query, top_k=5):

    temp_df = df.copy()

    if "vegan" in query.lower():
        temp_df = temp_df[
            temp_df["dietary_tags"]
            .str.contains("vegan", case=False, na=False)
        ]

    price_match = re.search(
        r"under\s+(\d+)",
        query.lower()
    )

    if price_match:
        limit = int(price_match.group(1))

        temp_df = temp_df[
            temp_df["price"] <= limit
        ]

    texts = temp_df["combined_text"].tolist()

    temp_embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True
    )

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    scores = cosine_similarity(
        query_embedding,
        temp_embeddings
    )[0]

    idx = np.argsort(scores)[::-1][:top_k]

    results = temp_df.iloc[idx].copy()

    results["similarity_score"] = scores[idx]

    return results

In [19]:
filtered_search(
    "vegan dishes under 200"
)

,dish_name,category,price,description,dietary_tags,combined_text,similarity_score
89,Green Salad,Salad,120,"A fresh medley of cucumber, tomato, onion, and...","Veg, Vegan","Green Salad | Salad | Veg, Vegan | ₹120 | A fr...",0.518991
72,Missi Roti,Bread,50,Whole wheat and gram flour flatbread spiced wi...,"Veg, Vegan","Missi Roti | Bread | Veg, Vegan | ₹50 | Whole ...",0.459056
10,Aloo Tikki,Starter,150,"Pan-fried potato patties spiced with cumin, co...","Veg, Vegan","Aloo Tikki | Starter | Veg, Vegan | ₹150 | Pan...",0.454451
74,Steamed Rice,Rice,150,"Plain steamed basmati rice, light and fluffy. ...","Veg, Vegan","Steamed Rice | Rice | Veg, Vegan | ₹150 | Plai...",0.453656
5,Veg Spring Rolls,Starter,200,Crispy fried rolls stuffed with finely shredde...,"Veg, Vegan","Veg Spring Rolls | Starter | Veg, Vegan | ₹200...",0.441165
